# FedMed FL Engine: Hospital A Client

This Google Colab notebook runs the federated learning client for **Hospital A**. It downloads the hospital's specific MSD Task01_BrainTumour split (or uses synthetic data as a fallback), performs local training using MONAI and a 3D U-Net, and uploads the TenSEAL homomorphic-encrypted weight deltas to the central Flower server.

## Step 1: System Check (Ensure GPU is active)

Go to **Runtime -> Change runtime type** and select **T4 GPU** (or any available GPU).

In [ ]:
import torch
gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Training 3D UNet will be extremely slow.")

## Step 2: Install Dependencies

Install Flower, MONAI, TenSEAL, and other packages required by the FedMed client.

In [ ]:
!pip install --upgrade pip
!pip install "flwr>=1.9.0,<2.0" tenseal>=0.3.14 monai[all]>=1.3.0 nibabel>=5.0 pyyaml>=6.0

## Step 3: Clone FedMed Codebase

We need to fetch the FedMed engine code to run the training client. Provide the Git repository URL below.

In [ ]:
# Replace with your actual repository URL if hosting privately
!git clone https://github.com/MayankJaint/FedMed.git
%cd fedmed
import sys
import os
sys.path.append(os.getcwd())

## Step 4: Configure Server URL and SILO Settings

Copy the **Cloudflare Tunnel URL** printed in the docker-compose output of the local server stack.
Keep `USE_SYNTHETIC = True` for a fast smoke test, or set to `False` to download and train on the real MSD brain tumor dataset split.

In [ ]:
SERVER_URL = "https://your-cloudflare-tunnel-url.trycloudflare.com" #@param {type:"string"}
HOSPITAL_ID = "hospital_a"
USE_SYNTHETIC = True #@param {type:"boolean"}

# Strip HTTP/HTTPS protocol prefixes
server_address = SERVER_URL.replace("https://", "").replace("http://", "").strip()

## Step 5: (Optional) Prepare Keys & Dataset Splits

If you are running in Homomorphic Encryption (`he_ckks`) mode, ensure that the public context key `keys/public.tenseal` exists. If you are using the real MSD dataset, prepare the hospital manifests first.

In [ ]:
# Run prep script to generate default hospital silo manifest splits
!python scripts/prepare_silos.py

# If HE keys directory is missing, create a placeholder public context for testing or download it from the server
import os
os.makedirs("keys", exist_ok=True)
if not os.path.exists("keys/public.tenseal"):
    print("HE public context key not found. If running in HE mode, upload it to keys/public.tenseal")

## Step 6: Start Flower Client

Connect to the central Flower orchestrator server. The client will wait until the minimum number of clients (configured on the server) are connected, and then begin training.

In [ ]:
import flwr as fl
from client.client import create_client

print(f"Launching {HOSPITAL_ID} Flower client connecting to: {server_address}")

fl.client.start_numpy_client(
    server_address=server_address,
    client=create_client(
        hospital_id=HOSPITAL_ID,
        use_synthetic=USE_SYNTHETIC,
    )
)